# Deceptron quick start

Run D-IPG (Deceptron Inverse-Preconditioned Gradient) with custom PyTorch maps, train a reverse map for a known
measurement equation, and display the pretrained Heat-3D recovery.

[Paper](https://arxiv.org/abs/2605.13068) · [Source code](https://github.com/AadityaKachhadiya/deceptron)

In [ ]:
%pip install "deceptron[demo] @ git+https://github.com/AadityaKachhadiya/deceptron.git"

## Solve an inverse problem!

The measurement is `y = 2x`. Here the reverse map is constructed explicitly.
D-IPG uses it to propose updates to the state.

In [ ]:
import torch
from deceptron import DIPG, DeceptronModule, SolverConfig
model = DeceptronModule(lambda x: 2 * x, lambda y: y / 2)
result = DIPG(model, torch.tensor([0.8]), torch.zeros(1), SolverConfig(tolerance_eps=1e-5))
print(result["x_hat"])  # approximately tensor([0.4000])

## Learn a reverse map

The forward map below takes two unknown state variables to three sensor
readings. Change `forward_map` to supply your own differentiable measurement
equation. The reverse map is a PyTorch model trained on synthetic paired
data. Its input and output dimensions match the measurements and state,
respectively.

In [ ]:
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from deceptron import TrainConfig, train_reverse, train_reverse_jcp

torch.manual_seed(7)
torch.set_num_threads(2)
device = torch.device("cpu")

def forward_map(x):
    u, v = x.unbind(dim=-1)
    return torch.stack((u + 0.25 * v, 0.5 * u + v, u - v), dim=-1)

states = torch.rand(120, 2)
x_train, x_val = states[:96], states[96:]
train = DataLoader(TensorDataset(x_train, forward_map(x_train)), batch_size=24, shuffle=True)
val = DataLoader(TensorDataset(x_val, forward_map(x_val)), batch_size=24)

model = DeceptronModule(forward_map, nn.Linear(3, 2))
config = TrainConfig(
    reverse_epochs=50, jcp_epochs=25,
    reverse_lr=0.03, jcp_lr=0.01, jcp_batch_subsample=4, eval_subset_rjcp=8,
)
model = train_reverse(model, train, val, x_val, config, device, verbose=False)
model = train_reverse_jcp(
    model, train, val, x_val, config, device, mlp_extra_losses=False, verbose=False,
)

truth = torch.tensor([0.25, 0.70])
measurement = forward_map(truth)
result = DIPG(model, measurement, torch.zeros_like(truth), SolverConfig(tolerance_eps=1e-4))
print("Recovered state:", result["x_hat"].tolist())
print("True state:", truth.tolist())
print(f"Measurement RMSE: {result['final_residual']:.6f}")

## Heat-3D reconstruction

Recover one held-out initial temperature field from a noisy measurement.
This cell downloads the example script and its archived checkpoint and
displays the generated figure below.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
from IPython.display import Image, display
import subprocess
import sys

source = "https://raw.githubusercontent.com/AadityaKachhadiya/deceptron/main/examples"
example = Path("heat3d_example")
(example / "assets").mkdir(parents=True, exist_ok=True)

for filename in ("heat3d_quickstart.py", "assets/heat3d_seed7_case07.pt"):
    urlretrieve(f"{source}/{filename}", example / filename)

figure = example / "heat3d_recovery.png"
subprocess.run(
    [sys.executable, str(example / "heat3d_quickstart.py"), "--output", str(figure)],
    check=True,
)
display(Image(filename=str(figure)))

## Paper experiments

The [reproduction guide](https://github.com/AadityaKachhadiya/deceptron/blob/main/reproduce/README.md)
links the seven benchmark scripts, archived data, and figure-generation commands.